# Лабораторная 03. Shuffle и `explain("formatted")`

Цель: научиться видеть shuffle в physical plan по оператору `Exchange`.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-03-shuffle-explain').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 17:06:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Spark UI: http://3bbcc89f06d1:4040


## Narrow operation
`filter/select` обычно не требует shuffle.

In [2]:
narrow_df = orders.filter(F.col('status') == 'paid').select('order_id', 'customer_id')
narrow_df.explain('formatted')

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [order_id#0L, customer_id#1L, status#3]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
PushedFilters: [IsNotNull(status), EqualTo(status,paid)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,status:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [order_id#0L, customer_id#1L, status#3]

(3) Filter [codegen id : 1]
Input [3]: [order_id#0L, customer_id#1L, status#3]
Condition : (isnotnull(status#3) AND (status#3 = paid))

(4) Project [codegen id : 1]
Output [2]: [order_id#0L, customer_id#1L]
Input [3]: [order_id#0L, customer_id#1L, status#3]




Вопрос: есть ли `Exchange`? Почему?

Ответ: потому что filter/select могут быть выполнены на отдельных партициях независимо

## groupBy
`groupBy` должен собрать одинаковые ключи вместе.

In [3]:
by_customer = orders.groupBy('customer_id').count()
by_customer.explain('formatted')
by_customer.count()

== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_id#1L]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<customer_id:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [customer_id#1L]

(3) HashAggregate [codegen id : 1]
Input [1]: [customer_id#1L]
Keys [1]: [customer_id#1L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#30L]
Results [2]: [customer_id#1L, count#31L]

(4) Exchange
Input [2]: [customer_id#1L, count#31L]
Arguments: hashpartitioning(customer_id#1L, 8), ENSURE_REQUIREMENTS, [plan_id=35]

(5) HashAggregate [codegen id : 2]
Input [2]: [customer_id#1L, count#31L]
Keys [1]: [customer_id#1L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#26L]
Results [2]: [customer_id#1L, count(1)#26L AS count#27L]




10000

Вопросы:

- Где появился `Exchange`? он появился после подсчета на отельных партиях, когда надо объединить результаты
- Почему `Exchange` связан с shuffle? потому что он обозначает обмен данными
- Сколько shuffle partitions задано? 8


## join
Отключим broadcast, чтобы увидеть shuffle join.

In [4]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
joined = orders.join(customers, 'customer_id')
joined.explain('formatted')
joined.count()

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,order_date:date,status:string,order_amount:decimal(10,2)>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]

(3) Filter [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Condition : isnotnull(customer_id#1L)

(4) Exchange
Input [5]: [order

120000

Вопросы:

- Какой join выбрал Spark? SortMergeJoin Inner
- Сколько `Exchange` в плане? 2
- Почему join часто требует shuffle? потому что нужно разделить обе таблицы по ключу, потом отсортирвоать и уже потом сливать


In [5]:
spark.stop()